# YOLOv8n 튜닝 모델 - Test Set 평가
- **파일명**: `11_test_evaluation_yolov8n_tuned.ipynb`
- **저장 경로**: `N:\개인\이수빈\3.13_Mini_Project\scripts\`
- **목적**: 하이퍼파라미터 튜닝한 YOLOv8n 모델의 Test set 평가
- **작성일**: 2026-03-04
- **평가 방식**: `model.predict(conf=threshold)` — YOLO 내부에서 threshold 필터링


## 셀 1: 라이브러리 import


In [1]:
# =============================================================================
# [셀 1] 라이브러리 임포트
# =============================================================================

# YOLO 모델을 불러오고 추론(예측)하는 라이브러리
from ultralytics import YOLO

# 폴더/파일 경로를 다루는 라이브러리 (os.path보다 안정적)
from pathlib import Path

# 이미지 처리 및 bbox(탐지 박스) 그리기용
import cv2

# 파일 복사용 (미탐/오탐 이미지 원본 보존)
import shutil

# 표 형태 데이터를 다루는 라이브러리
import pandas as pd

# 결과를 JSON 파일로 저장하기 위한 라이브러리
import json

# 현재 날짜/시간을 기록하기 위한 라이브러리
from datetime import datetime

print("✅ 라이브러리 import 완료")


✅ 라이브러리 import 완료


## 셀 2: 경로 설정


In [2]:
# =============================================================================
# [셀 2] 프로젝트 경로 설정
# =============================================================================

# 프로젝트 최상위 폴더 경로 (본인 PC에 맞게 수정)
PROJECT_ROOT = Path(r'N:\개인\이수빈\3.13_Mini_Project')

# 튜닝된 모델 가중치 파일 경로 (학습 중 가장 좋았던 모델)
MODEL_PATH = PROJECT_ROOT / 'results' / 'yolov8n_tuned' / 'weights' / 'best.pt'

# 테스트 이미지가 들어있는 폴더 경로
# fire/ 폴더와 normal/ 폴더로 나뉘어 있음 (폴더명이 정답 역할)
TEST_DIR = PROJECT_ROOT / 'DATASET' / 'test_set_894'

# 평가 결과를 저장할 폴더 경로
EVAL_DIR = PROJECT_ROOT / 'evaluation' / 'yolov8n_tuned_test'

# 미탐/오탐 이미지 저장용 하위 폴더 생성
(EVAL_DIR / 'false_negatives').mkdir(parents=True, exist_ok=True)            # 미탐 (bbox 표시)
(EVAL_DIR / 'false_negatives' / 'originals').mkdir(parents=True, exist_ok=True)  # 미탐 원본
(EVAL_DIR / 'false_positives').mkdir(parents=True, exist_ok=True)            # 오탐 (bbox 표시)
(EVAL_DIR / 'false_positives' / 'originals').mkdir(parents=True, exist_ok=True)  # 오탐 원본

print("=" * 60)
print("📊 YOLOv8n Tuned Test Set 평가")
print("=" * 60)
print(f"\n모델: {MODEL_PATH}")
print(f"Test: {TEST_DIR}")
print(f"결과: {EVAL_DIR}")


📊 YOLOv8n Tuned Test Set 평가

모델: N:\개인\이수빈\3.13_Mini_Project\results\yolov8n_tuned\weights\best.pt
Test: N:\개인\이수빈\3.13_Mini_Project\DATASET\test_set_894
결과: N:\개인\이수빈\3.13_Mini_Project\evaluation\yolov8n_tuned_test


## 셀 3: TDD 테스트 함수 정의


In [3]:
# =============================================================================
# [셀 3] TDD - 테스트 함수 먼저 정의
# TDD란? 코드를 만들기 전에 "이 코드가 통과해야 할 조건"을 먼저 정하는 방법론
# =============================================================================

def test_model_loaded(model):
    """테스트 1: 모델이 정상적으로 로드되었는지 확인"""
    assert model is not None, "모델 로드 실패!"
    print("✅ 테스트 1 통과: 모델 정상 로드됨")

def test_test_images_exist(fire_images, non_fire_images):
    """테스트 2: 테스트 이미지가 제대로 있는지 확인"""
    assert len(fire_images) > 0, "화재 이미지가 없습니다!"
    assert len(non_fire_images) > 0, "비화재(normal) 이미지가 없습니다!"
    total = len(fire_images) + len(non_fire_images)
    assert total > 0, f"테스트 이미지 총 {total}장 — 0장이면 안 됩니다!"
    print(f"✅ 테스트 2 통과: 화재 {len(fire_images)}장, 비화재 {len(non_fire_images)}장, 총 {total}장")

def test_evaluation_metrics(recall, precision, f1):
    """테스트 3: 평가 지표가 유효한 범위(0~1)인지 확인"""
    assert 0 <= recall <= 1, f"Recall 값 이상: {recall}"
    assert 0 <= precision <= 1, f"Precision 값 이상: {precision}"
    assert 0 <= f1 <= 1, f"F1 값 이상: {f1}"
    print("✅ 테스트 3 통과: 평가 지표 정상 범위")

def test_kpi_check(recall, precision):
    """테스트 4: KPI 목표 달성 여부 확인"""
    recall_pass = recall >= 0.920
    precision_pass = precision >= 0.880
    print(f"{'✅' if recall_pass else '❌'} Recall: {recall:.3f} (목표: 0.920)")
    print(f"{'✅' if precision_pass else '❌'} Precision: {precision:.3f} (목표: 0.880)")
    if recall_pass and precision_pass:
        print("🎉 KPI 목표 달성!")
    else:
        print("⚠️ KPI 목표 미달성")


## 셀 4: 모델 로드 및 테스트 1 실행


In [4]:
# =============================================================================
# [셀 4] 모델 로드 및 테스트 1 실행
# =============================================================================

print("\n[모델 로드]")

# 튜닝된 YOLOv8n 모델을 불러옴 (best.pt = 학습 중 가장 좋았던 시점)
model = YOLO(str(MODEL_PATH))

# 테스트 1 실행
test_model_loaded(model)



[모델 로드]
✅ 테스트 1 통과: 모델 정상 로드됨


## 셀 5: 테스트 이미지 수집 (대소문자 무시, 중복 제거)


In [5]:
# =============================================================================
# [셀 5] 테스트 이미지 목록 수집 (대소문자 무시, 중복 제거)
# set을 사용해서 .jpg와 .JPG가 같은 파일을 2번 세는 것을 방지
# =============================================================================

# 화재 이미지 수집 — set으로 중복 방지
fire_paths = set()  # set: 같은 값이 들어오면 자동으로 무시하는 자료구조
for ext in ['*.jpg', '*.jpeg', '*.png', '*.webp',
            '*.JPG', '*.JPEG', '*.PNG', '*.WEBP']:
    for p in (TEST_DIR / 'fire').glob(ext):
        fire_paths.add(p)  # set에 추가 (이미 있으면 무시됨)

# 정상 이미지 수집 — 폴더명 'normal' 사용
normal_paths = set()
for ext in ['*.jpg', '*.jpeg', '*.png', '*.webp',
            '*.JPG', '*.JPEG', '*.PNG', '*.WEBP']:
    for p in (TEST_DIR / 'normal').glob(ext):
        normal_paths.add(p)

# Path 객체를 문자열로 변환 + 정렬 (재현성 확보)
test_fire = sorted(list(fire_paths))      # 화재 이미지 리스트
test_normal = sorted(list(normal_paths))  # 정상 이미지 리스트

# 수집 결과 출력
print(f"\n수집된 이미지:")
print(f"  화재: {len(test_fire)}장")
print(f"  정상: {len(test_normal)}장")
print(f"  총: {len(test_fire) + len(test_normal)}장")

# 테스트 2 실행 (문자열 변환 버전으로 전달)
fire_images_str = [str(p) for p in test_fire]
normal_images_str = [str(p) for p in test_normal]
test_test_images_exist(fire_images_str, normal_images_str)



수집된 이미지:
  화재: 447장
  정상: 447장
  총: 894장
✅ 테스트 2 통과: 화재 447장, 비화재 447장, 총 894장


## 셀 6: Test 평가 실행
- `model.predict(conf=CONF_THRESHOLD)` 사용
- YOLO 내부에서 threshold 이하 탐지를 필터링
- 미탐/오탐 이미지는 bbox 표시하여 별도 저장


In [6]:
# =============================================================================
# [셀 6] Test 평가 실행
# 핵심: model.predict(conf=0.20)으로 YOLO 내부에서 threshold 필터링
# =============================================================================

print("\n[평가 시작]")
print(f"시작: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Confidence threshold 설정
# 모델이 "화재일 확률 20% 이상"이면 화재로 판정
CONF_THRESHOLD = 0.20

# ============================================
# 1. 화재 이미지 평가
# ============================================
# TP(True Positive): 실제 화재 → 화재로 탐지 (정탐)
# FN(False Negative): 실제 화재 → 놓침 (미탐) ⚠️ 가장 위험!
true_positive = 0   # 정탐 카운터
false_negative = 0  # 미탐 카운터
fn_list = []        # 미탐 이미지 경로 저장

print(f"\n[화재 이미지 평가 중...]")

for idx, img_path in enumerate(test_fire, 1):
    # model.predict에 conf를 직접 전달 → YOLO가 내부에서 threshold 적용
    # 이렇게 하면 conf 미만의 약한 탐지가 결과에서 자동 제외됨
    results = model.predict(
        str(img_path),         # 이미지 경로 (문자열로 변환)
        conf=CONF_THRESHOLD,   # ★ 핵심: threshold를 YOLO에 직접 전달
        imgsz=640,             # 이미지 크기 640x640
        save=False,            # 자동 저장 안 함 (우리가 직접 저장)
        verbose=False          # 로그 출력 최소화
    )

    # 탐지된 bbox(바운딩 박스) 개수 확인
    num_boxes = len(results[0].boxes)

    if num_boxes > 0:
        # 화재를 화재로 맞춤 → TP (정탐) ✅
        true_positive += 1
    else:
        # 화재를 놓침 → FN (미탐) ⚠️
        false_negative += 1
        fn_list.append(img_path)

        # 미탐 이미지에 "NOT DETECTED" 텍스트 표시하여 저장
        img = cv2.imread(str(img_path))  # 이미지 로드
        cv2.putText(
            img, 'NOT DETECTED',         # 표시할 텍스트
            (50, 50),                    # 텍스트 위치 (x, y)
            cv2.FONT_HERSHEY_SIMPLEX,    # 폰트
            1.5, (0, 0, 255), 3          # 크기, 빨강색(BGR), 두께
        )
        # 텍스트 표시된 이미지 저장
        save_path = EVAL_DIR / 'false_negatives' / f"FN_{img_path.name}"
        cv2.imwrite(str(save_path), img)
        # 원본도 복사 (비교 분석용)
        orig_path = EVAL_DIR / 'false_negatives' / 'originals' / img_path.name
        shutil.copy2(img_path, orig_path)

    # 진행 상황 (50장마다 출력)
    if idx % 50 == 0:
        print(f"  {idx}/{len(test_fire)}")

print(f"✅ 화재 평가 완료")
print(f"   탐지: {true_positive}장")
print(f"   미탐: {false_negative}장")

# ============================================
# 2. 정상 이미지 평가 (오탐 확인)
# ============================================
# TN(True Negative): 정상 → 정상으로 맞춤 (정탐)
# FP(False Positive): 정상 → 화재로 오인 (오탐)
true_negative = 0   # 정탐 카운터
false_positive = 0  # 오탐 카운터
fp_list = []        # 오탐 이미지 경로 저장

print(f"\n[정상 이미지 평가 중...]")

for idx, img_path in enumerate(test_normal, 1):
    # 정상 이미지도 동일하게 model.predict(conf=threshold)로 평가
    results = model.predict(
        str(img_path),         # 이미지 경로
        conf=CONF_THRESHOLD,   # ★ 동일한 threshold 적용
        imgsz=640,             # 이미지 크기
        save=False,            # 자동 저장 안 함
        verbose=False          # 로그 최소화
    )

    # 탐지된 bbox 개수 확인
    num_boxes = len(results[0].boxes)

    if num_boxes > 0:
        # 정상을 화재로 잘못 판정 → FP (오탐)
        false_positive += 1
        fp_list.append(img_path)

        # bbox가 그려진 이미지 생성 (어디를 화재로 봤는지 확인용)
        result_img = results[0].plot()  # bbox, label, conf 자동 표시
        save_path = EVAL_DIR / 'false_positives' / f"FP_{img_path.name}"
        cv2.imwrite(str(save_path), result_img)
        # 원본도 복사
        orig_path = EVAL_DIR / 'false_positives' / 'originals' / img_path.name
        shutil.copy2(img_path, orig_path)
    else:
        # 정상을 정상으로 맞춤 → TN (정탐) ✅
        true_negative += 1

    # 진행 상황 (50장마다 출력)
    if idx % 50 == 0:
        print(f"  {idx}/{len(test_normal)}")

print(f"✅ 정상 평가 완료")
print(f"   정상: {true_negative}장")
print(f"   오탐: {false_positive}장")

print(f"\n평가 완료: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)



[평가 시작]
시작: 2026-03-04 11:43:58

[화재 이미지 평가 중...]
  50/447
  100/447
  150/447
  200/447
  250/447
  300/447
  350/447
  400/447
✅ 화재 평가 완료
   탐지: 412장
   미탐: 35장

[정상 이미지 평가 중...]
  50/447
  100/447
  150/447
  200/447
  250/447
  300/447
  350/447
  400/447
✅ 정상 평가 완료
   정상: 423장
   오탐: 24장

평가 완료: 2026-03-04 11:45:34


## 셀 7: 성능 지표 계산 및 결과 출력


In [7]:
# =============================================================================
# [셀 7] 성능 지표 계산 및 결과 출력
# =============================================================================

print("\n[성능 지표 계산]")

# Recall = TP / (TP + FN)
# 의미: 실제 화재 중 몇 %를 잡아냈는가?
recall = true_positive / (true_positive + false_negative) if (true_positive + false_negative) > 0 else 0

# Precision = TP / (TP + FP)
# 의미: "화재다"라고 한 것 중 실제로 몇 %가 진짜인가?
precision = true_positive / (true_positive + false_positive) if (true_positive + false_positive) > 0 else 0

# F1 Score = 2 × (Precision × Recall) / (Precision + Recall)
# 의미: Recall과 Precision의 조화 평균
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

# Accuracy = (TP + TN) / 전체
accuracy = (true_positive + true_negative) / (true_positive + true_negative + false_positive + false_negative)

# ── TDD 테스트 3, 4 실행 ──
test_evaluation_metrics(recall, precision, f1_score)
test_kpi_check(recall, precision)

# ── 결과 출력 ──
print(f"\n{'='*60}")
print("📊 YOLOv8n Tuned - Test Set 최종 성능")
print(f"{'='*60}")

# 혼동행렬
print(f"\n혼동 행렬:")
print(f"                실제 화재    실제 정상")
print(f"  예측 화재    {true_positive:>6}       {false_positive:>6}  ")
print(f"  예측 정상    {false_negative:>6}       {true_negative:>6}  ")

# 주요 지표
print(f"\n주요 지표:")
print(f"  Recall:    {recall:.3f}  (목표: 0.920)")
print(f"  Precision: {precision:.3f}  (목표: 0.880)")
print(f"  F1 Score:  {f1_score:.3f}")
print(f"  Accuracy:  {accuracy:.3f}")

# 상세 분석
print(f"\n상세 분석:")
print(f"  총 화재:   {len(test_fire)}장")
print(f"    탐지:    {true_positive}장 ({true_positive/len(test_fire)*100:.1f}%)")
print(f"    미탐:    {false_negative}장 ({false_negative/len(test_fire)*100:.1f}%) ⚠️")
print(f"\n  총 정상:   {len(test_normal)}장")
print(f"    정상:    {true_negative}장 ({true_negative/len(test_normal)*100:.1f}%)")
print(f"    오탐:    {false_positive}장 ({false_positive/len(test_normal)*100:.1f}%)")

# 목표 달성 확인
print(f"\n{'='*60}")
print("🎯 목표 달성 여부")
print(f"{'='*60}")
if recall >= 0.92:
    print(f"✅ Recall 달성! {recall:.3f} (목표 0.920, +{(recall-0.92)*100:.1f}%p)")
else:
    print(f"⚠️ Recall 부족: {recall:.3f} (목표 0.920, {(0.92-recall)*100:.1f}%p 부족)")
if precision >= 0.88:
    print(f"✅ Precision 달성! {precision:.3f} (목표 0.880, +{(precision-0.88)*100:.1f}%p)")
else:
    print(f"⚠️ Precision 부족: {precision:.3f} (목표 0.880, {(0.88-precision)*100:.1f}%p 부족)")
print(f"{'='*60}")



[성능 지표 계산]
✅ 테스트 3 통과: 평가 지표 정상 범위
✅ Recall: 0.922 (목표: 0.920)
✅ Precision: 0.945 (목표: 0.880)
🎉 KPI 목표 달성!

📊 YOLOv8n Tuned - Test Set 최종 성능

혼동 행렬:
                실제 화재    실제 정상
  예측 화재       412           24  
  예측 정상        35          423  

주요 지표:
  Recall:    0.922  (목표: 0.920)
  Precision: 0.945  (목표: 0.880)
  F1 Score:  0.933
  Accuracy:  0.934

상세 분석:
  총 화재:   447장
    탐지:    412장 (92.2%)
    미탐:    35장 (7.8%) ⚠️

  총 정상:   447장
    정상:    423장 (94.6%)
    오탐:    24장 (5.4%)

🎯 목표 달성 여부
✅ Recall 달성! 0.922 (목표 0.920, +0.2%p)
✅ Precision 달성! 0.945 (목표 0.880, +6.5%p)


## 셀 8: Default 모델과 비교


In [8]:
# =============================================================================
# [셀 8] Default vs 튜닝 모델 비교 (Threshold 0.20 기준)
# =============================================================================

print(f"\n{'='*60}")
print("📈 Default vs 튜닝 모델 비교 (Threshold 0.20 기준)")
print(f"{'='*60}")
print(f"{'항목':<12} {'Default':>10} {'Tuned':>10} {'차이':>10}")
print("-" * 42)
print(f"{'Recall':<12} {'0.915':>10} {recall:>10.3f} {recall - 0.915:>+10.3f}")
print(f"{'Precision':<12} {'0.965':>10} {precision:>10.3f} {precision - 0.965:>+10.3f}")
print(f"{'F1':<12} {'0.939':>10} {f1_score:>10.3f} {f1_score - 0.939:>+10.3f}")
print(f"{'미탐(FN)':<12} {'38':>10} {false_negative:>10} {false_negative - 38:>+10}")
print(f"{'오탐(FP)':<12} {'15':>10} {false_positive:>10} {false_positive - 15:>+10}")
print(f"{'='*60}")



📈 Default vs 튜닝 모델 비교 (Threshold 0.20 기준)
항목              Default      Tuned         차이
------------------------------------------
Recall            0.915      0.922     +0.007
Precision         0.965      0.945     -0.020
F1                0.939      0.933     -0.006
미탐(FN)               38         35         -3
오탐(FP)               15         24         +9


## 셀 9: 결과 저장 (JSON + CSV)


In [9]:
# =============================================================================
# [셀 9] 결과를 JSON + CSV로 저장
# =============================================================================

# ── JSON 저장 ──
results_data = {
    "model": "YOLOv8n_tuned",
    "threshold": CONF_THRESHOLD,
    "evaluation_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "dataset": {
        "fire_images": len(test_fire),
        "normal_images": len(test_normal),
        "total": len(test_fire) + len(test_normal)
    },
    "confusion_matrix": {
        "TP": true_positive,
        "FN": false_negative,
        "FP": false_positive,
        "TN": true_negative
    },
    "metrics": {
        "recall": round(recall, 4),
        "precision": round(precision, 4),
        "f1_score": round(f1_score, 4),
        "accuracy": round(accuracy, 4)
    },
    "kpi_check": {
        "recall_target": 0.920,
        "recall_pass": recall >= 0.920,
        "precision_target": 0.880,
        "precision_pass": precision >= 0.880
    }
}

# JSON 파일 저장
json_path = EVAL_DIR / f"test_results_th{str(CONF_THRESHOLD).replace('.','')}.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(results_data, f, indent=2, ensure_ascii=False)
print(f"💾 JSON 저장: {json_path}")

# ── 미탐(FN) CSV 저장 ──
if fn_list:
    fn_data = [{"file": p.name, "path": str(p)} for p in fn_list]
    fn_df = pd.DataFrame(fn_data)
    fn_csv = EVAL_DIR / f"fn_list_th{str(CONF_THRESHOLD).replace('.','')}.csv"
    fn_df.to_csv(fn_csv, index=False, encoding="utf-8-sig")
    print(f"💾 미탐 목록: {fn_csv} ({len(fn_list)}건)")

# ── 오탐(FP) CSV 저장 ──
if fp_list:
    fp_data = [{"file": p.name, "path": str(p)} for p in fp_list]
    fp_df = pd.DataFrame(fp_data)
    fp_csv = EVAL_DIR / f"fp_list_th{str(CONF_THRESHOLD).replace('.','')}.csv"
    fp_df.to_csv(fp_csv, index=False, encoding="utf-8-sig")
    print(f"💾 오탐 목록: {fp_csv} ({len(fp_list)}건)")

print("\n✅ 모든 결과 저장 완료!")


💾 JSON 저장: N:\개인\이수빈\3.13_Mini_Project\evaluation\yolov8n_tuned_test\test_results_th02.json
💾 미탐 목록: N:\개인\이수빈\3.13_Mini_Project\evaluation\yolov8n_tuned_test\fn_list_th02.csv (35건)
💾 오탐 목록: N:\개인\이수빈\3.13_Mini_Project\evaluation\yolov8n_tuned_test\fp_list_th02.csv (24건)

✅ 모든 결과 저장 완료!


## 셀 10: 최종 요약 및 다음 단계


In [10]:
# =============================================================================
# [셀 10] 최종 요약
# =============================================================================

print(f"\n{'='*60}")
print("📝 최종 요약")
print(f"{'='*60}")
print(f"1. 튜닝 모델 Test Recall: {recall:.3f} (Default: 0.915)")
print(f"2. 튜닝 모델 Test Precision: {precision:.3f} (Default: 0.965)")
print(f"3. 미탐 이미지: {false_negative}건 → evaluation/yolov8n_tuned_test/false_negatives/")
print(f"4. 오탐 이미지: {false_positive}건 → evaluation/yolov8n_tuned_test/false_positives/")

print(f"\n[다음 단계 판단]")
if recall > 0.915 and precision > 0.880:
    print("→ 튜닝 모델이 Default보다 Recall 개선됨!")
    print("→ 약한 튜닝(scale 조정) 재시도 가치 있음")
elif recall >= 0.920 and precision >= 0.880:
    print("→ 튜닝 모델이 KPI 달성!")
    print("→ 추가 튜닝 없이 이 모델 사용 가능")
else:
    print("→ 튜닝 모델이 Default보다 성능 하락")
    print("→ 약한 튜닝(scale 중심)으로 재시도하거나 Default 확정")
print(f"{'='*60}")



📝 최종 요약
1. 튜닝 모델 Test Recall: 0.922 (Default: 0.915)
2. 튜닝 모델 Test Precision: 0.945 (Default: 0.965)
3. 미탐 이미지: 35건 → evaluation/yolov8n_tuned_test/false_negatives/
4. 오탐 이미지: 24건 → evaluation/yolov8n_tuned_test/false_positives/

[다음 단계 판단]
→ 튜닝 모델이 Default보다 Recall 개선됨!
→ 약한 튜닝(scale 조정) 재시도 가치 있음
